# Ispezione Casuale dei Cluster (Campionamento Stratificato)

Questo notebook seleziona 100 email in modo stratificato dai cluster generati e ne esporta il contenuto testuale (`content_clean` e `subject_clean`) in un file markdown per una validazione qualitativa umana.

Siccome il file dei cluster contiene solo gli ID di assegnazione, lo uniremo al file `processed` per recuperare il testo originale.

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Paths
CLUSTER_PATH = Path("../../data/processed/jmail_emails_clustered.parquet")
PROCESSED_PATH = Path("../../data/processed/jmail_emails_processed.parquet")
LABELS_PATH = Path("../../data/metadata/cluster_labeling_metadata.json")
REPORT_PATH = Path("../../reports/100_random_emails_inspection.md")

if not CLUSTER_PATH.exists():
    CLUSTER_PATH = Path("../../data/processed/jmail_emails_clustered_soft.parquet")


In [2]:
print("Caricamento dati...")
df_clusters = pd.read_parquet(CLUSTER_PATH)
df_processed = pd.read_parquet(PROCESSED_PATH, columns=['id', 'subject_clean', 'combined_text'])

print(f"Record clusterizzati: {len(df_clusters)}")
print(f"Record processati: {len(df_processed)}")

# Uniamo i dataframe sull'id univoco per recuperare il testo
df = df_clusters.merge(df_processed, on='id', how='left')
print(f"Record dopo il merge: {len(df)}")

Caricamento dati...
Record clusterizzati: 1757624
Record processati: 1757624
Record dopo il merge: 1757624


In [3]:
cluster_names = {}
if LABELS_PATH.exists():
    with open(LABELS_PATH, "r") as f:
        labels_meta = json.load(f)
    for k, v in labels_meta.items():
        try:
            cluster_id = int(k)
            cluster_names[cluster_id] = v.get("llm_summary", "Sconosciuto")
        except ValueError:
            pass

# Se la colonna 'cluster_id' non esiste, cerchiamo 'cluster'
col_cluster = 'cluster_id' if 'cluster_id' in df.columns else 'cluster'

df['cluster_name'] = df[col_cluster].map(lambda x: cluster_names.get(x, f"Topic {x}"))

## Campionamento Stratificato
Selezioniamo 100 righe, estraendo equamente dai vari cluster per diversificare il campione.

In [5]:
NUM_SAMPLES = 100

if len(df) <= NUM_SAMPLES:
    df_sample = df.copy()
else:
    # Drop righe con id nullo o cluster non definito, se presenti
    df = df.dropna(subset=[col_cluster, 'combined_text'])
    
    cluster_counts = df[col_cluster].value_counts()
    weights = df[col_cluster].map(lambda x: 1.0 / cluster_counts[x])
    
    df_sample = df.sample(n=NUM_SAMPLES, weights=weights, random_state=42)

df_sample = df_sample.sort_values(by=[col_cluster])
print(f"Campionate {len(df_sample)} email da {df_sample[col_cluster].nunique()} cluster differenti.")

Campionate 100 email da 53 cluster differenti.


## Generazione Report

In [7]:
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    f.write("# Ispezione Casuale Cluster Email\n\n")
    f.write("Questo documento contiene un campione stratificato di email per valutare la coerenza dei cluster.\n\n")
    
    for idx, row in df_sample.iterrows():
        c_id = row.get(col_cluster, 'N/A')
        c_name = row.get('cluster_name', 'Sconosciuto')
        
        content = row.get('combined_text', '')
        if not isinstance(content, str) or not content.strip():
            content = "[TESTO VUOTO O NON DISPONIBILE]"
            
        f.write("---\n\n")
        f.write(f"### Cluster {c_id}: {c_name}\n")
        
        subject = row.get('subject_clean', '')
        if isinstance(subject, str) and subject.strip():
            f.write(f"**Oggetto**: {subject}\n\n")
        
        f.write("**Contenuto (`combined_text`)**: \n\n")
        f.write(f"> {content.replace('\n', '\n> ')}\n\n")

print(f"File di report generato con successo in: {REPORT_PATH}")

The one and only
So what do you

Want to get back to mgmt and bwg- Tom will get u going again- but when u come back-u need to stay with us- as you will be a mrg and be for real- so we will be counting on you Sent from Steve Hanson's Blackberry - Proud to be the first national multi-concept restaurant group to be certified Green by the Green Restaurant Association - Confidentiality Notice: This e-mail transmission and any file or previous e-mail attached to it is intended to be viewed only by the party to which it is addressed and may contain valuable business infonnation that is confidential and/or otherwise protected from disclosure under applicable law. If you am not the intended recipient you are hereby notified that any review, disclosure, dissemination or use of any of the information contained in or attached to this transmission is STRICTLY PROHIBITED. Thank you for your cooperation.
Re: Any shot on later today or is sunday better

come now On Sat, Sep 21, 2013 at 3:30 PM, Steve 